In [1]:
# import moviepy
# from moviepy import *
# from moviepy.video import *
import copy
'''
# loading video gfg
clip = VideoFileClip("../Data/modelUnderwaters.mp4")


# getting subclip from it
clip = clip.subclipped(0, 4)

# applying speed effect
effect = vfx.AccelDecel(1, abruptness=0, soonness=1)

clip = effect.apply(clip)



# showing final clip
#clip.display_in_notebook()


#from conf import SAMPLE_INPUTS, SAMPLE_OUTPUTS

length = 10

clip1 = VideoFileClip("../Data/modelUnderwaters.mp4").subclipped(0, 0 + 5)
clip2 = VideoFileClip("../Data/flipturnHQ.mp4").subclipped(0, 0 + 5)


combined = clips_array([[clip1], [clip2]])

effect = vfx.Margin(left=960, right=960)
combined = effect.apply(combined)
effect = vfx.Resize((1920,1080) )
combined = effect.apply(combined)

#combined = combined.resize(height=1920)


effect = vfx.Crop(x1=1166.6, y1=0, x2=2246.6, y2=1920)
#clip = effect.apply(clip)
#combined = combined.crop(x1=1166.6, y1=0, x2=2246.6, y2=1920)

#combined = combined.with_effects(vfx.Resize((1920,1080)), vfx.Margin(960))
combined.display_in_notebook
#combined.write_videofile('test.mp4')

combined.write_videofile("../outputs/result.mp4")

'''

'\n# loading video gfg\nclip = VideoFileClip("../Data/modelUnderwaters.mp4")\n\n\n# getting subclip from it\nclip = clip.subclipped(0, 4)\n\n# applying speed effect\neffect = vfx.AccelDecel(1, abruptness=0, soonness=1)\n\nclip = effect.apply(clip)\n\n\n\n# showing final clip\n#clip.display_in_notebook()\n\n\n#from conf import SAMPLE_INPUTS, SAMPLE_OUTPUTS\n\nlength = 10\n\nclip1 = VideoFileClip("../Data/modelUnderwaters.mp4").subclipped(0, 0 + 5)\nclip2 = VideoFileClip("../Data/flipturnHQ.mp4").subclipped(0, 0 + 5)\n\n\ncombined = clips_array([[clip1], [clip2]])\n\neffect = vfx.Margin(left=960, right=960)\ncombined = effect.apply(combined)\neffect = vfx.Resize((1920,1080) )\ncombined = effect.apply(combined)\n\n#combined = combined.resize(height=1920)\n\n\neffect = vfx.Crop(x1=1166.6, y1=0, x2=2246.6, y2=1920)\n#clip = effect.apply(clip)\n#combined = combined.crop(x1=1166.6, y1=0, x2=2246.6, y2=1920)\n\n#combined = combined.with_effects(vfx.Resize((1920,1080)), vfx.Margin(960))\ncombin

In [2]:
import accelerate
from pathlib import Path

#print(accelerate.__version__)

c:\Users\danre\Documents\GitHub\SwimVisionProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:


import torch
import numpy as np
import cv2
import matplotlib
import argparse
import os
import time
from PIL import Image
from transformers import (
    AutoProcessor,
    RTDetrForObjectDetection,
    VitPoseForPoseEstimation,
)
print(torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
index = 0
OUT_DIR = '../outputs'

while True:
    save =  'Trial - ' + str(index)
    outputPath = Path(f'{OUT_DIR}/{save}')
    if outputPath.is_dir():
        index += 1
        continue
    else:
        os.makedirs(f'{OUT_DIR}/{save}', exist_ok=False)
        OUT_DIR = OUT_DIR + f'/{save}'
        break


model_name = 'usyd-community/vitpose-plus-huge'



#save_name = 'Ivan'#args.input.split(os.path.sep)[-1].split('.')[0]
# Define codec and create VideoWriter object.

# Load detector.
person_image_processor = AutoProcessor.from_pretrained(
    'PekingU/rtdetr_r50vd_coco_o365'
)
person_model = RTDetrForObjectDetection.from_pretrained(
    'PekingU/rtdetr_r50vd_coco_o365', device_map=device
)
# Load ViTPose.
#print(f"Pose Model: {'../models/VitPose-s_RePoGen.pth'}")
image_processor = AutoProcessor.from_pretrained(model_name)
model = VitPoseForPoseEstimation.from_pretrained(pretrained_model_name_or_path=model_name, device_map=device)


#This is sadly chatted :(
# modelCustom = torch.load('../Models/VitPose-s_RePoGen.pth', device, weights_only=False)
# model.load_state_dict(modelCustom)
# image_processor.load_state_dict(modelCustom)
# image_processor.eval()
# model.eval()

True


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [4]:
import math

edges = [
    (0, 1), (0, 2), (2, 4), (1, 3), (6, 8), (8, 10),
    (5, 7), (7, 9), (5, 11), (11, 13), (13, 15), (6, 12),
    (12, 14), (14, 16), (5, 6), (11, 12)
]

def lowToHigh(inputList):
    if (inputList[0] < inputList[1]):
        return inputList
    else:
        return (inputList[1], inputList[0])

def detect_objects(image):
    """
    :param image: Image in PIL image format.
    Returns:
        person_boxes: Bboxes of persons in [x, y, w, h] format.
    """
    det_time_start = time.time()
    inputs = person_image_processor(
        images=image, return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        outputs = person_model(**inputs)
    target_sizes = torch.tensor([(image.height, image.width)])
    
    results = person_image_processor.post_process_object_detection(
        outputs, target_sizes=target_sizes, threshold=0.2
    )
    det_time_end = time.time()
    det_fps = 1 / (det_time_end-det_time_start)
    
    # Extract the first result, as we can pass multiple images at a time.
    result = results[0]
    
    # In COCO dataset, humans labels have index 0.
    person_boxes_xyxy = result['boxes'][result['labels'] == 0]
    person_boxes_xyxy = person_boxes_xyxy.cpu().numpy()
    
    # Convert boxes from (x1, y1, x2, y2) to (x1, y1, w, h) format.
    person_boxes = person_boxes_xyxy.copy()
    person_boxes[:, 2] = person_boxes[:, 2] - person_boxes[:, 0]
    person_boxes[:, 3] = person_boxes[:, 3] - person_boxes[:, 1]
    return person_boxes, det_fps
def detect_pose(image, person_boxes):
    """
    :param image: Image in PIL image format.
    :param person_bboxes: Batched person boxes in [[x, y, w, h], ...] format.
    """
    pose_time_start = time.time()
    inputs = image_processor(
        image, boxes=[person_boxes], return_tensors='pt'
    ).to(device)
    
    dataset_index = torch.tensor([0], device=device) # must be a tensor of shape (batch_size,)
    if len(person_boxes) != 0:
        if 'plus' in model_name:
            with torch.no_grad():
                outputs = model(**inputs, dataset_index=dataset_index)
        else:
            with torch.no_grad():
                outputs = model(**inputs)
        
        pose_results = image_processor.post_process_pose_estimation(
            outputs, boxes=[person_boxes]
        )
    pose_time_end = time.time()
    pose_fps = 1 / (pose_time_end-pose_time_start)
    if len(person_boxes) == 0:
        return [], pose_fps
    image_pose_result = pose_results[0]
    
    return image_pose_result, pose_fps
def draw_keypoints(outputs, image):
    """
    :param outputs: Outputs from the keypoint detector.
    :param image: Image in PIL Image format.
    Returns:
        image: Annotated image Numpy array format.
    """
    image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    # the `outputs` is list which in-turn contains the dictionaries 
    for i, pose_result in enumerate(outputs):
        keypoints = pose_result['keypoints'].cpu().detach().numpy()
        # proceed to draw the lines if the confidence score is above 0.9
        keypoints = keypoints[:, :].reshape(-1, 2)
        if keypoints.shape[0] != 0 and keypoints.shape[0] != 17:
            print(keypoints.shape[0]) 
        for p in range(keypoints.shape[0]):
            # draw the keypoints
            cv2.circle(image, (int(keypoints[p, 0]), int(keypoints[p, 1])), 
                        3, (0, 0, 255), thickness=-1, lineType=cv2.FILLED)
            # uncomment the following lines if you want to put keypoint number
            cv2.putText(image, f'{p}', (int(keypoints[p, 0]+10), int(keypoints[p, 1]-5)),
                         cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
        for ie, e in enumerate(edges):
            # get different colors for the edges
            rgb = matplotlib.colors.hsv_to_rgb([
                ie/float(len(edges)), 1.0, 1.0
            ])
            rgb = rgb*255
            # join the keypoint pairs to draw the skeletal structure
            cv2.line(image, (int(keypoints[e, 0][0]), int(keypoints[e, 1][0])),
                    (int(keypoints[e, 0][1]), int(keypoints[e, 1][1])),
                    tuple(rgb), 2, lineType=cv2.LINE_AA)
        break
    return image


#Whole skeleton
# body_edges = [(6, 8), (8, 10),
#     (5, 7), (7, 9), (5, 11), (11, 13), (6, 12), (13, 15),
#     (12, 14), (14, 16), (5, 6), (11, 12)]

#Underwaters from left only
body_edges = [(5, 7), (7, 9), (5, 11), (11, 13), (13, 15)] #THESE NEED TO BE LOWER NUMBER FIRST FOR ANGLE OPERATIONS TO WORK


#https://stackoverflow.com/questions/14066933/direct-way-of-computing-the-clockwise-angle-between-two-vectors
# [left, vertex, right]
# Format needs to be this way because it calculates clockwise angle, not minimum angle
angles =[[11, 5, 7], [13, 11, 5], [11, 13, 15]] # Placing the reference vectors in the vertical may prove a liability

def getAngles(points):
    angleList = []
    if points:
        for i in range(len(angles)):
            curEdge1 = lowToHigh((angles[i][0], angles[i][1]))
            curEdge2 = lowToHigh((angles[i][1], angles[i][2]))
            print(curEdge1)
            print(curEdge2)
            index1 = body_edges.index(curEdge1)
            index2 = body_edges.index(curEdge2)
            print(f"Index {body_edges[index1][0]}, index {body_edges[index1][1]}")
            print(f"Point 1: {points[body_edges[index1][0]][0]}, Points 2: {points[body_edges[index1][1]]}")
            print(f"Norm1x: {points[index1][0] - points[angles[i][1]][0]}, Norm1y: {points[index1][1] - points[angles[i][1]][1]}") #Error in which points it calls here
            print(f"Norm2x: {points[index2][0] - points[angles[i][1]][0]}, Norm2y: {points[index2][1] - points[angles[i][1]][1]}") 
            print(f"EIndex {index1}, Index {index2}")
            print(f'EFWIMF{points[angles[i][1]][0], points[angles[i][1]][1]}')
            print(f"Point 1: {points[index1]}, Points 2: {points[index2]}")
            print(f"{points[body_edges[index1][0]][0]} - {points[angles[i][1]][0]}")
            x1 = points[body_edges[index1][0]][0] - points[angles[i][1]][0]
            y1 = points[body_edges[index1][0]][1] - points[angles[i][1]][1]
            x2 = points[body_edges[index2][0]][0] - points[angles[i][1]][0]
            y2 = points[body_edges[index2][0]][1] - points[angles[i][1]][1]
            print(f'{x1}, {y1}    {x2}, {y2}')
            pointCombo = [[points[body_edges[index1][0]][0] - points[angles[i][1]][0], points[body_edges[index1][0]][1] - points[angles[i][1]][1]], [points[body_edges[index2][0]][0] - points[angles[i][1]][0], points[body_edges[index2][0]][1] - points[angles[i][1]][1]]]
            print(pointCombo)
            print(calcAngle(pointCombo))
            angleList.append(calcAngle(pointCombo))
    return angleList


def calcAngle(pointSet):
    p1, p2 = pointSet
    dot = p1[0] * p2[0] + p1[1] * p2[1]
    det = p1[0] * p2[1] - p1[1] * p2[0] # Could have botched this ngl
    rawAngle = math.atan2(det, dot) + math.pi
    return math.degrees(rawAngle)


print(calcAngle([[0.7, -.7], [-0.7, -0.7]]))


samplePoints = [(12, -34), #0
(-8, 41), #1
(25, 25), #2
(-47, -10), #3
(3, 19), #4
(33, -4), #5
(-21, -39), #6
(45, 12), #7
(-15, 28), #8
(0, -48), #9
(37, 37), #10
(-29, 5), #11
(18, -22), #12
(-1, -1), #13
(40, -40), #14
(-35, 15), #15
(10, 48)] #16

getAngles(samplePoints)


def drawAngles(frame, image, angleSet, points, height):
    if points:
        points = switchOpenCVNormal( image, points)
        for point in points:
            print(point)
        for i in range(len(angleSet)):
            leftIndex = angles[i][0]
            vertexIndex = angles[i][1]
            rightIndex = angles[i][2]

            leftPoint = points[leftIndex]
            vertexPoint = points[vertexIndex]
            rightPoint = points[rightIndex]


            initialPoint = getAverageCoord(getAverageCoord(leftPoint, vertexPoint), getAverageCoord(vertexPoint, rightPoint))

            if (angleSet[i] < 90):
                drawPoint = initialPoint
            else:
                slopeLine = slope(vertexPoint, initialPoint)
                if (angleSet[i] < 180):
                    if leftPoint[0] >= rightPoint[0]:
                        if leftPoint[1] >= rightPoint[1]:
                            drawPoint = [initialPoint[0] + math.cos(math.radians(angleSet[i]))*slopeLine*10, initialPoint[1] - math.sin(math.radians(angleSet[i]))*slopeLine* 10]
                        else:
                            drawPoint = [initialPoint[0] - math.cos(math.radians(angleSet[i]))*slopeLine*10, initialPoint[1] - math.sin(math.radians(angleSet[i]))*slopeLine* 10]
                    else:
                        if leftPoint[1] >= rightPoint[1]:
                            drawPoint = [initialPoint[0] + math.cos(math.radians(angleSet[i]))*slopeLine*10, initialPoint[1] + math.sin(math.radians(angleSet[i]))*slopeLine* 10]
                        else:
                            drawPoint = [initialPoint[0] - math.cos(math.radians(angleSet[i]))*slopeLine*10, initialPoint[1] + math.sin(math.radians(angleSet[i]))*slopeLine* 10]
                else:
                    if leftPoint[0] >= rightPoint[0]:
                        if leftPoint[1] >= rightPoint[1]:
                            drawPoint = [initialPoint[0] - math.cos(math.radians(angleSet[i]))*slopeLine*10, initialPoint[1] + math.sin(math.radians(angleSet[i]))*slopeLine* 10]
                        else:
                            drawPoint = [initialPoint[0] + math.cos(math.radians(angleSet[i]))*slopeLine*10, initialPoint[1] + math.sin(math.radians(angleSet[i]))*slopeLine* 10]
                    else:
                        if leftPoint[1] >= rightPoint[1]:
                            drawPoint = [initialPoint[0] - math.cos(math.radians(angleSet[i]))*slopeLine*10, initialPoint[1] - math.sin(math.radians(angleSet[i]))*slopeLine* 10]
                        else:
                            drawPoint = [initialPoint[0] + math.cos(math.radians(angleSet[i]))*slopeLine*10, initialPoint[1] - math.sin(math.radians(angleSet[i]))*slopeLine* 10]

            cv2.putText(frame, f'{round(angleSet[i], 2)}', (int(drawPoint[0]), int(height - drawPoint[1])), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (240, 19, 255), 1) # I converted to left-hand coordinates here
    return frame

#Need to figure out how to display reflexive angle








def getKeypoints(outputs):
    out = []
    if outputs:
        keypoints = outputs[0]['keypoints'].cpu().detach().numpy()
        keypoints = keypoints[:, :].reshape(-1, 2)
        for p in range(keypoints.shape[0]):
            out.append([int(keypoints[p, 0]), int(keypoints[p, 1])])
    return out

def drawSlopes(image, slopes, points):
    if slopes and points:
        for i, edge in enumerate(body_edges):
            average = getAverageCoord(points[edge[0]], points[edge[1]])
            cv2.putText(image, f'{round(slopes[i], 2)}', (average[0] -10, average[1]),
                             cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)           
    return image



def getSlopes(outputs, image, points):
    slopes = []
    if outputs and points:
        for i, edge in enumerate(body_edges):
            slopes.append(slope(points[edge[0]], points[edge[1]]))
    return slopes


def switchOpenCVNormal(image, kps):
    newkps = copy.deepcopy(kps)
    print(newkps == kps)
    width, height = image.size
    for i in range(len(newkps)):
        newkps[i][1] = height - newkps[i][1]
    return newkps

def getAverageCoord(coord1, coord2):
    return [int((coord1[0] + coord2[0])/2), int((coord1[1] + coord2[1])/2)]

def slope(coord1, coord2):
    if coord1[0] - coord2[0] != 0:
        return ((coord1[1] - coord2[1]) / (coord1[0] - coord2[0])) 
    else:
        return 100 #Arbitrarily high slope


90.0
(5, 11)
(5, 7)
Index 5, index 11
Point 1: 33, Points 2: (-29, 5)
Norm1x: -8, Norm1y: 29
Norm2x: -21, Norm2y: -30
EIndex 2, Index 0
EFWIMF(33, -4)
Point 1: (25, 25), Points 2: (12, -34)
33 - 33
0, 0    0, 0
[[0, 0], [0, 0]]
180.0
(11, 13)
(5, 11)
Index 11, index 13
Point 1: -29, Points 2: (-1, -1)
Norm1x: -18, Norm1y: -15
Norm2x: 54, Norm2y: 20
EIndex 3, Index 2
EFWIMF(-29, 5)
Point 1: (-47, -10), Points 2: (25, 25)
-29 - -29
0, 0    62, -9
[[0, 0], [62, -9]]
180.0
(11, 13)
(13, 15)
Index 11, index 13
Point 1: -29, Points 2: (-1, -1)
Norm1x: -46, Norm1y: -9
Norm2x: 4, Norm2y: 20
EIndex 3, Index 4
EFWIMF(-1, -1)
Point 1: (-47, -10), Points 2: (3, 19)
-29 - -1
-28, 6    0, 0
[[-28, 6], [0, 0]]
180.0


In [5]:


def makeMainVideo(videoName):
    inputPath = '../Data/' + videoName + '.mp4'
    cap = cv2.VideoCapture(inputPath)
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    video_fps = int(cap.get(5)) 
    
    frame_count = 0 # To count total frames.
    total_fps = 0 # To get the final frames per second.
    masterSlopeList = []


    os.makedirs(f'{OUT_DIR}/{videoName}', exist_ok=False)
    out = cv2.VideoWriter(
        f'{OUT_DIR}/{videoName}/MainVideo.mp4', 
        cv2.VideoWriter_fourcc(*'mp4v'), 
        video_fps, 
        (frame_width, frame_height), 
    )

    if not cap.isOpened():
        print("Not opening")
        
    while cap.isOpened():
        ret, frame = cap.read()
        if ret:
            #frame = cv2.flip(frame, 1)
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            image = Image.fromarray(frame_rgb)

            start_time = time.time()
            bboxes, det_fps = detect_objects(image=image)
            image_pose_result, pose_fps = detect_pose(image=image, person_boxes=bboxes)
            result = draw_keypoints(image_pose_result, image)
            kps = getKeypoints(image_pose_result)
            normalkps = switchOpenCVNormal(image, kps)
            listSlopes = getSlopes(image_pose_result, image, normalkps)
            result = drawSlopes(result, listSlopes, kps)
            angle = getAngles(kps)
            result = drawAngles(result, image, angle, normalkps, frame_height)
            
            
            
            masterSlopeList.append(listSlopes)
            end_time = time.time()
            forward_pass_time = end_time - start_time
                
            # Get the current fps.
            fps = 1 / (forward_pass_time)
            # Add `fps` to `total_fps`.
            total_fps += fps
            # Increment frame count.
            frame_count += 1
            cv2.putText(
                result,
                f"Frame Count: {frame_count-1} | FPS: {fps:0.1f} | Pose FPS: {pose_fps:0.1f} | Detection FPS: {det_fps:0.1f}",
                (15, 25),
                fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                fontScale=1.0,
                color=(0, 0, 255),
                thickness=2,
                lineType=cv2.LINE_AA,
            )
            out.write(result)
            cv2.imshow('Prediction', result)
            # Press `q` to exit
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        else:
            break
    cap.release()
    cv2.destroyAllWindows()
    out.release()
    avg_fps = total_fps/frame_count
    print(f"Average FPS: {avg_fps}")
    return masterSlopeList



 # Release VideoCapture().

# Close all frames and video windows.



In [6]:
# print(masterSlopeList)
# print(len(masterSlopeList))
# print(frame_count)

In [7]:
def percentDifference (v1, v2):
    return abs(v1-v2) / ((v1 + v2) / 2) * 100

jointComboId = 4

def findHigh(slopeList, initial):
    if slopeList[initial]:
        lastSlope = slopeList[initial][jointComboId]
    for i in range (initial, len(slopeList)):
        print(f"CHECKING FRAME {i}")
        if slopeList[i]:
            curSlope = slopeList[i][jointComboId]
            if (curSlope < lastSlope):
                return i 
            lastSlope = curSlope
            end = i
    return end-1

def findLow(slopeList, initial):
    if slopeList[initial]:
        lastSlope = slopeList[initial][jointComboId]
    for i in range (initial, len(slopeList)):
        print(f"CHECKING FRAME {i}")
        if slopeList[i]:
            curSlope = slopeList[i][jointComboId]
            if (curSlope > lastSlope):
                return i  
            lastSlope = curSlope
            end = i
    return end -1


def makeClips(videoName, slopeList, kicks):
    kickFrames = []
    start = 0
    for i in range(kicks):
        initial = findHigh(slopeList, start)
        print(initial)
        end = findLow(slopeList, initial)
        print(end)
        kickFrames.append([initial, end])
        start = end

    print(kickFrames)

    # fileName = 'ShortUnderwaters.mp4'
    # inputPath = '../Data/' + fileName
    # cap = cv2.VideoCapture(inputPath)
    os.makedirs(f'{OUT_DIR}/{videoName}/Clips')
    for frameCombo in kickFrames:
        inputPath = '../Data/' + videoName + '.mp4'
        cap2 = cv2.VideoCapture(inputPath)
        #cap2 = cv2.VideoCapture(f'{OUT_DIR}/{videoName}/MainVideo.mp4')  OLD WITH COLORS
        if not cap2.isOpened():
            print("Error: Could not open video file.")
        else:
            print("Video file opened successfully!")
        index = 0
        frame_width = int(cap2.get(3))
        frame_height = int(cap2.get(4))
        video_fps = int(cap2.get(5)) 
        while True:
            clip_save_index = str(index)
            outputPath = Path(f'{OUT_DIR}/{videoName}/Clips/{clip_save_index}.mp4')
            if outputPath.is_file():
                index += 1
                continue
            else:
                out2 = cv2.VideoWriter(
                    f'{OUT_DIR}/{videoName}/Clips/{clip_save_index}.mp4', 
                    cv2.VideoWriter_fourcc(*'mp4v'), 
                    video_fps, 
                    (frame_width, frame_height), 
                )
                index = index + 1
                break
        
        for i in range(frameCombo[0], frameCombo[1]):
            cap2.set(cv2.CAP_PROP_POS_FRAMES, i)
            res, frame = cap2.read()
            if res:
                #cv2.imshow('Frame', frame)
                out2.write(frame)
        #print(f'{OUT_DIR}/clipOutputs/{save_name}.mp4')
        cap2.release()
        out2.release()
        cv2.destroyAllWindows()

    
    

# Close all frames and video windows.

 

Agenda for tmrw:

1. Throw the slope gathering into a method (done)
2. Make clip creation a method (done)
3. pick minimum amount of kicks and compare the first n kicks (done)
4. Create clips, make sure the learner clips are normalized to the expert's length (doing)
5. Calc difference every single slope between each frame of the normalized clips.  Store best and worse absolute diff + average for that frame
6. Create a list for each frame that records diff of each edge
7. Make a method to turn  diff list into summary list with 
 [average, [best joint, accuracy], [worst frame, accuracy]]
8. Make methods to extract this data to find overall superlatives (best frame, worst frame, worst joint of them all, best joint of them all)
9. Get accuracy for each kick and overall
10. If you really want splice the kick clips tg, get acc % differences for each joint every frame of recording, color grade them based on % diff, extend clip to like 10 seconds 

In [8]:

Mainslopes1 = makeMainVideo("OtherUnderwater")
makeClips("OtherUnderwater", Mainslopes1, 4)

Mainslopes2 = makeMainVideo("ShortUnderwater")
makeClips("ShortUnderwater", Mainslopes2, 3)

def getClipSlopes(videoName, number, learner):
    if learner:
        inputPath = f'{OUT_DIR}/{videoName}/Clips/{number}N.mp4'
    else:
        inputPath = f'{OUT_DIR}/{videoName}/Clips/{number}.mp4'
    cap = cv2.VideoCapture(inputPath)
    # frame_width = int(cap.get(3))
    # frame_height = int(cap.get(4))
    # video_fps = int(cap.get(5)) 
    
    frame_count = 0 # To count total frames.
    total_fps = 0 # To get the final frames per second.
    slopesList = []
    if not cap.isOpened():
        print("Not opening")
        
    while cap.isOpened():
        ret, frame = cap.read()
        if ret:
            #frame = cv2.flip(frame, 1)
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = Image.fromarray(frame_rgb)
            start_time = time.time()
            bboxes, det_fps = detect_objects(image=image)
            image_pose_result, pose_fps = detect_pose(image=image, person_boxes=bboxes)
            #result = draw_keypoints(image_pose_result, image)
            kps = getKeypoints(image_pose_result)
            normalkps = switchOpenCVNormal(image, kps)
            listSlopes = getSlopes(image_pose_result, image, normalkps)
            #result = drawSlopes(result, listSlopes, kps)
            slopesList.append(listSlopes)
            end_time = time.time()
            forward_pass_time = end_time - start_time
                
            # Get the current fps.
            fps = 1 / (forward_pass_time)
            # Add `fps` to `total_fps`.
            total_fps += fps
            # Increment frame count.
            frame_count += 1
            '''cv2.putText(
                result,
                f"Frame Count: {frame_count-1} | FPS: {fps:0.1f} | Pose FPS: {pose_fps:0.1f} | Detection FPS: {det_fps:0.1f}",
                (15, 25),
                fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                fontScale=1.0,
                color=(0, 0, 255),
                thickness=2,
                lineType=cv2.LINE_AA,
            )'''
            #out.write(result)
            #cv2.imshow('Prediction', result)
            # Press `q` to exit
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        else:
            break
    cap.release()
    cv2.destroyAllWindows()
    #out.release()
    avg_fps = total_fps/frame_count
    print(f"Average FPS: {avg_fps}")
    return slopesList

# #Deprecated moviepy version
# def normalizeLearnersClips(learnersVid, expertsVid):
#     #assuming we get least amount of kicks somewhere
    
#     for i in range(3):
#         learnerClip = moviepy.VideoFileClip(f'{OUT_DIR}/{learnersVid}/Clips/{i}.mp4')
#         expertClip = moviepy.VideoFileClip(f'{OUT_DIR}/{expertsVid}/Clips/{i}.mp4')

#         targetDur = expertClip.duration * 10
#         #combined = combined.with_effects(vfx.Resize((1920,1080)), vfx.Margin(960))
#         learnerClip = learnerClip.with_effects([vfx.AccelDecel(targetDur * 3)])
#         expertClip = expertClip.with_effects([vfx.AccelDecel(targetDur * 3)])
#         #I might have to delete the clip before
#         learnerClip.write_videofile(f'{OUT_DIR}/{learnersVid}/Clips/{i}.mp4', codec='libx264')
#         expertClip.write_videofile(f'{OUT_DIR}/{expertsVid}/Clips/{i}.mp4', codec='libx264')

        

#Opencv version

def normalizeLearnersClips(learnersVid, expertsVid):
    #assuming we get least amount of kicks somewhere
    for i in range(3):
        capExpert = cv2.VideoCapture(f'{OUT_DIR}/{expertsVid}/Clips/{i}.mp4')
        targetLength = int(capExpert.get(cv2.CAP_PROP_FRAME_COUNT))
        print(targetLength)
        capExpert.release()
        capLearner = cv2.VideoCapture(f'{OUT_DIR}/{learnersVid}/Clips/{i}.mp4')
        currentLength = int(capLearner.get(cv2.CAP_PROP_FRAME_COUNT))
        if targetLength - currentLength == 0:
            interval = 0
        else:
            interval = currentLength / (targetLength - currentLength)

        print(f'\nTarget length: {targetLength}\nCurrentLength: {currentLength}\nInterval: {interval}\n')
        frameCount = 0
        frame_width = int(capLearner.get(3))
        frame_height = int(capLearner.get(4))
        video_fps = int(capLearner.get(5))
        count = interval
        out3 = cv2.VideoWriter(
            f'{OUT_DIR}/{learnersVid}/Clips/{i}N.mp4', 
            cv2.VideoWriter_fourcc(*'mp4v'), 
            video_fps, 
            (frame_width, frame_height), 
        )
        while capLearner.isOpened():
            ret, frame = capLearner.read()
            if ret:
                frameCount = frameCount + 1
                if interval == 0:
                    out3.write(frame)
                    print(f"Frame {frameCount}: normal frame")
                else: 

                    print(f'Count: {count}, Rounded: {round(count)}, frameCount: {frameCount}, interval: {interval}, ')
                    if (round(count) == frameCount):
                        out3.write(frame) 
                        out3.write(frame)
                        print(f"Frame {frameCount}: double Frame")
                        count = count + interval
                    elif (-1*(round(count)) == frameCount):
                        print(f"Frame {frameCount}: frame Skipped")
                        count = count + interval
                    else:
                        print(f"Frame {frameCount}: normal frame")
                        out3.write(frame)
                    # if round(frameCount * interval) == : #Double frame
                    #     out3.write(frame) 
                    #     out3.write(frame)
                    #     print(f"Frame {frameCount}: double Frame")
                    # elif frameCount % (-1*interval) == 0:
                    #     print(f"Frame {frameCount}: frame Skipped")
                    # else:
                    #     print(f"Frame {frameCount}: normal frame")
                    #     out3.write(frame)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
            else:
                break
        capLearner.release()
        out3.release()
        cv2.destroyAllWindows()
        print("all done!")
        del capLearner
        del out3





def compareClip(learnersVid, expertsVid, kicks):
    normalizeLearnersClips(learnersVid, expertsVid)
    masterlist = []
    #Starting the clip access phase
    for k in range(kicks):
        diffSlopesList = []
        learnerSlopes = getClipSlopes(learnersVid, k, True)
        expertSlopes = getClipSlopes(expertsVid, k, False)
        outputList = []
        for i in range(len(learnerSlopes)):
            diffList = []
            if learnerSlopes[i] and expertSlopes[i]:
                for j in range(len(learnerSlopes[i])): #I think it's just 17
                    diff = learnerSlopes[i][j] - expertSlopes[i][j]
                    diffList.append(diff)
                    outputList.append([f"F{i} E{j}", diff])
                diffSlopesList.append(diffList)
        sorted(outputList, key=lambda x: (x[1])) #THIS IS THE ERROR CULPRIT
        for elem in outputList:
            print(f'{elem[0]}: {elem[1]}')
        masterlist.append(outputList)
    return masterlist

#compareClip("OtherUnderwater", "ShortUnderwater", 0)
    

master = compareClip("OtherUnderwater", "ShortUnderwater", 3)

print(master)
    





True
(5, 11)
(5, 7)
Index 5, index 11
Point 1: 1035, Points 2: [1320, 787]
Norm1x: -64, Norm1y: 72
Norm2x: -51, Norm2y: 75
EIndex 2, Index 0
EFWIMF(1035, 723)
Point 1: [971, 795], Points 2: [984, 798]
1035 - 1035
0, 0    0, 0
[[0, 0], [0, 0]]
180.0
(11, 13)
(5, 11)
Index 11, index 13
Point 1: 1320, Points 2: [1495, 781]
Norm1x: -343, Norm1y: -25
Norm2x: -349, Norm2y: 8
EIndex 3, Index 2
EFWIMF(1320, 787)
Point 1: [977, 762], Points 2: [971, 795]
1320 - 1320
0, 0    -285, -64
[[0, 0], [-285, -64]]
180.0
(11, 13)
(13, 15)
Index 11, index 13
Point 1: 1320, Points 2: [1495, 781]
Norm1x: -518, Norm1y: -19
Norm2x: -518, Norm2y: 1
EIndex 3, Index 4
EFWIMF(1495, 781)
Point 1: [977, 762], Points 2: [977, 782]
1320 - 1495
-175, 6    0, 0
[[-175, 6], [0, 0]]
180.0
True
[984, 798]
[971, 777]
[971, 795]
[977, 762]
[977, 782]
[1035, 723]
[1027, 769]
[926, 747]
[928, 772]
[825, 768]
[833, 776]
[1320, 787]
[1304, 827]
[1495, 781]
[1481, 791]
[1588, 687]
[1575, 696]
True
(5, 11)
(5, 7)
Index 5, index 1